In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
import json

# --- STEP 1: LOAD MODEL ---
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Ananda100/qweenlora",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-05 03:57:40.914194: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772683061.136035      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772683061.192801      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772683061.688128      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772683061.688163      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772683061.688165      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.75G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen3-8b-base-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


adapter_model.safetensors:   0%|          | 0.00/349M [00:00<?, ?B/s]

Unsloth 2026.3.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [3]:
# --- STEP 2: PREPARE DATASET ---
from datasets import load_dataset

# 1. Define your prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        # Format the text and add the EOS token
        text = alpaca_prompt.format(inst, inp, out) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

# 2. LOAD SEPARATE FILES DIRECTLY
# Replace these paths with the actual locations of your split files
dataset = load_dataset("json", data_files={
    "train": "/kaggle/input/datasets/adarsha100/trainingset/training.json", 
    "test":  "/kaggle/input/datasets/adarsha100/testingset/testing.json"
})

# 3. Apply the mapping to both sets
train_dataset = dataset["train"].map(formatting_prompts_func, batched = True)
eval_dataset  = dataset["test"].map(formatting_prompts_func, batched = True)

# Keep a raw copy of test data for manual evaluation/scoring later
eval_dataset_raw = dataset["test"]

print(f"Dataset loaded. Train size: {len(train_dataset)}, Test size: {len(eval_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset loaded. Train size: 9000, Test size: 1000


In [4]:
!pip install evaluate sacrebleu rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00


In [5]:
from tqdm import tqdm
from evaluate import load
from bert_score import score as bert_score_fn
import math
import torch

def run_evaluation(model, tokenizer, dataset, title="Model"):
    print(f"\n🚀 Starting Evaluation: {title}")
    total_samples = len(dataset)

    # Load metrics
    bleu = load("bleu")
    chrf = load("chrf")
    rouge = load("rouge")

    predictions, references, total_loss = [], [], 0.0

    # Using enumerate for the sample counter
    for i, row in enumerate(tqdm(dataset, desc="Processing")):
        # 1. Generate Response (Hidden from print)
        prompt = alpaca_prompt.format(row['instruction'], row['input'], "")
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        # We use a modest max_new_tokens to speed up base model testing
        outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
        resp = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split("### Response:")[-1].strip()

        predictions.append(resp)
        references.append(row['output'])

        # 2. Perplexity Calculation
        full_text = alpaca_prompt.format(row['instruction'], row['input'], row['output'])
        encodings = tokenizer(full_text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            loss = model(**encodings, labels=encodings["input_ids"]).loss
            total_loss += loss.item()

        # 3. Progress Print (Shows count only)
        # Using end="\r" keeps it on one line to keep the console clean
        print(f"✅ Sample {i+1}/{total_samples} done", end="\r")

    # 4. Final Calculations
    print(f"\n\n🧮 Calculating final scores for {title}...")

    # Perplexity Formula: $PPL = e^{(\frac{1}{N}\sum loss)}$
    ppl = math.exp(total_loss / total_samples)

    # BERTScore
    _, _, b_f1 = bert_score_fn(predictions, references, lang="bert-base-multilingual-cased")

    # Compute Standard Metrics
    rouge_results = rouge.compute(predictions=predictions, references=references)
    bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])
    chrf_results = chrf.compute(predictions=predictions, references=[[r] for r in references], word_order=2)

    results = {
        "PPL": ppl,
        "BERT": b_f1.mean().item(),
        "BLEU": bleu_results['bleu'],
        "chrF++": chrf_results['score'],
        "ROUGE-1": rouge_results['rouge1'],
        "ROUGE-2": rouge_results['rouge2'],
        "ROUGE-L": rouge_results['rougeL']
    }

    print(f"✨ Evaluation Complete for {title}!")
    for metric, value in results.items():
        print(f"🔹 {metric}: {value:.4f}")

    return results

# Get Baseline
base_results = run_evaluation(model, tokenizer, eval_dataset_raw, "Base Model")


🚀 Starting Evaluation: Base Model


Processing:   0%|          | 1/1000 [00:29<8:15:56, 29.79s/it]

Processing:   0%|          | 2/1000 [00:41<5:15:49, 18.99s/it]

Processing:   0%|          | 3/1000 [00:51<4:12:40, 15.21s/it]

Processing:   0%|          | 4/1000 [00:54<2:50:02, 10.24s/it]

...

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

✨ Evaluation Complete for Base Model!
🔹 PPL: 2.9463
🔹 BERT: 0.7505
🔹 BLEU: 0.0550
🔹 chrF++: 27.4712
🔹 ROUGE-1: 0.2915
🔹 ROUGE-2: 0.1162
🔹 ROUGE-L: 0.2511
